# Minimal example: from a far-field file to coupling rates

This notebook walks through the post-processing modules of the toolbox on one dataset:
a 500 nm Si slab on a mirror with a hyperuniform-disordered pillar pattern (Figs. 6e and 7 of the paper),
binned in steps of 0.5 um^-1.

The input is the summary file written by `02. Far field Transform/farfield_power_analysis.lsf` after an
FDTD run (module 1-2). Everything below needs only numpy, scipy and matplotlib (`pip install -e .` in the
repository root). It runs in about a minute.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

from mctoolbox.io import read_farfield, load_nk
from mctoolbox.heatmap import power_fractions
from mctoolbox.fit import KFitSettings, EFitSettings, fit_kspace, fit_espace, analyse_coupling, lorentzian, cmt_absorptance
from mctoolbox.units import k0, light_line
from mctoolbox.plotting.style import apply_style

apply_style(usetex=False, fontsize=11)
DATA = Path('..')/'figures'/'fig7'

## 1. Read the far-field output

Each row of the file is one wavelength. `read_farfield` turns it into rows of
`[photon energy (eV), k_parallel (um^-1), power fraction]`, normalised to the power that enters the
structure (`norm='R'`, i.e. 1 - R).

In [ ]:
ff = read_farfield(DATA/'500nmSi-wAbs-HUDPillar-H175R150nm-BS5.0um-Acc4-8M4400nmfit-lambda413-1127_V23.fsp_0.5um1_full_out_v128.dat', norm='R')
si = load_nk(DATA/'siliconR.txt', DATA/'siliconI.txt')
print(ff.rows.shape, 'rows;', len(ff.energies), 'photon energies; k step', ff.k_step, 'um^-1')

## 2. Energy-momentum heatmap (module 3)

Power per momentum bin versus photon energy. Guided modes live between the air light line (white) and the
silicon light line (edge of the black region).

In [ ]:
E, k, P = ff.rows.T
fig, ax = plt.subplots(figsize=(7, 4))
ax.fill_between(ff.energies, 0, light_line(ff.energies, si.n_um), color='black')
sc = ax.scatter(E, k, c=100*P, s=2, vmax=3, cmap='viridis')
ax.plot(ff.energies, k0(ff.energies), color='w', lw=0.7)
ax.set(xlabel='Photon energy (eV)', ylabel=r'$k_\parallel$ ($\mu$m$^{-1}$)', xlim=(1.1, 3), ylim=(0, 60))
fig.colorbar(sc, label='Power (%)');

## 3. Escape cone versus trapped light (module 4)

Summing the bins below and above the air light line k0 gives the fraction of the power that can leave the
slab again and the fraction coupled beyond the escape cone.

In [ ]:
e, within, beyond = power_fractions(ff)
fig, ax = plt.subplots(figsize=(7, 3))
ax.stackplot(e, beyond, within, labels=['beyond $k_c$', 'within $k_c$'], colors=['#7FA0CE', '#EBEBEB'])
ax.set(xlabel='Photon energy (eV)', ylabel='Power fraction', xlim=(1.1, 3), ylim=(0, 1)); ax.legend(loc='upper left');

## 4. Guided-mode resonances in the momentum domain (modules 5-6)

A slice at fixed photon energy shows GMR peaks in k. `fit_kspace` finds the peaks beyond the air light line
and fits Lorentzians: the center is the in-plane momentum of the resonance, the width its FWHM_m / 2.

In [ ]:
E_slice = 1.247
sl = ff.at_energy(E_slice)
fits, peaks = fit_kspace(sl[0, 0], sl[:, 1], sl[:, 2], KFitSettings())
kk = np.linspace(sl[:, 1].min(), sl[:, 1].max(), 2000)
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(sl[:, 1], 100*sl[:, 2], lw=2, label=f'E = {sl[0, 0]:.3f} eV')
for c, a, w in fits:
    ax.plot(kk, 100*lorentzian(kk, c, a, w), '--', color='grey')
ax.set(xlabel=r'$k_\parallel$ ($\mu$m$^{-1}$)', ylabel='P (%)', xlim=(5, 30)); ax.legend()
print(np.round(fits, 4))

## 5. Coupling rates in the energy domain (module 6)

At fixed k_parallel, a GMR follows temporal coupled-mode theory (Eq. HH of the paper),
A(E) = gamma_i gamma_e / ((E - E0)^2 + (gamma_i + N gamma_e)^2 / 4).

For N = 1 this shape is symmetric in gamma_i and gamma_e: an over-coupled (gamma_e/gamma_i = r) and an
under-coupled (1/r) resonance look identical. The default `branch='bulk-bound'` keeps the branch whose
internal loss rate gamma_i does not exceed the bulk absorption rate of silicon.

In [ ]:
k_slice = 20.0
sl = ff.at_momentum(k_slice)
fits = fit_espace(sl[:, 0], sl[:, 2], si, EFitSettings())
EE = np.linspace(sl[:, 0].min(), sl[:, 0].max(), 4000)
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(sl[:, 0], 100*sl[:, 2], lw=2, label=rf'$k_\parallel$ = {k_slice} $\mu$m$^{{-1}}$')
for e0, amp, gi, ge, N, unique in fits:
    ax.plot(EE, 100*cmt_absorptance(EE, e0, gi, ge, N), '--', color='grey')
    print(f'E0 = {e0:.4f} eV   gamma_i = {1e6*gi:.1f} ueV   gamma_e = {1e6*ge:.1f} ueV   gamma_e/gamma_i = {ge/gi:.0f}')
ax.set(xlabel='Photon energy (eV)', ylabel='P (%)', xlim=(1.1, 1.4)); ax.legend();

## 6. All resonances at once

`analyse_coupling` repeats both fits over the whole dataset (here up to 1.35 eV, as in Fig. 7).

In [ ]:
ff_low = read_farfield(DATA/'500nmSi-wAbs-HUDPillar-H175R150nm-BS5.0um-Acc4-8M4400nmfit-lambda413-1127_V23.fsp_0.5um1_full_out_v128.dat',
                       norm='R', max_energy=1.35)
res = analyse_coupling(ff_low, si, KFitSettings(), EFitSettings(k_grid=(10, 25.5, 0.5)))
t = res.espace_table()   # k, E0, amplitude, gamma_i, gamma_e, N, unique, gamma_e/gamma_i
fig, ax = plt.subplots(figsize=(6, 4))
sc = ax.scatter(t[:, 1], t[:, 0], c=np.log10(t[:, 7]), cmap='coolwarm', vmin=-3, vmax=3)
ax.set(xlabel='Photon energy (eV)', ylabel=r'$k_\parallel$ ($\mu$m$^{-1}$)')
fig.colorbar(sc, label=r'log$_{10}$($\gamma_e/\gamma_i$)')
print(f'{len(t)} resonances; median gamma_e/gamma_i = {np.median(t[:, 7]):.0f}')